# Can we predict 12-month loan default?
Methodology follows `checklist.md`.

**Data facts / assumptions**
- `application_id` is unique per row, so every row is its own entity (no grouping needed).
- There is no date column, and `application_id` is not ordered, so no temporal split is possible. We assume the model will score new applications drawn from the same population and use a random, stratified split. If row order is in fact chronological this would be optimistic.
- `application_id` is an identifier and is dropped. All other features are recorded at application, so are known at prediction time; `defaulted` is the target.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss, confusion_matrix
SEED=0
df=pd.read_csv('data.csv')
assert df.application_id.is_unique
y=df.pop('defaulted'); df=df.drop(columns='application_id')
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']; cat=['region']
print(df.shape, y.mean())

## 1. Hold out the test set (before any modelling)
20% of rows, stratified on the target. Entities are unique rows, so whole entities are held out.

In [ ]:
X_tr,X_te,y_tr,y_te=train_test_split(df,y,test_size=0.2,stratify=y,random_state=SEED)
print(len(X_tr),len(X_te),y_tr.mean(),y_te.mean())

## 2-4. Candidates (named before scoring) and CV design
Candidates, fixed in advance:
1. **baseline**: predict the training prevalence (trivial).
2. **logreg**: standardised numerics + one-hot region, L2 logistic regression, C in {0.01, 0.1, 1, 10}.
3. **hgb**: HistGradientBoosting, (max_depth, learning_rate) in {2,3}x{0.03,0.1}, 200 iterations, no early stopping.

Features: all 7 non-identifier columns for every model (no feature search). CV: 5-fold stratified on training rows only (rows are unique entities); preprocessing lives inside the Pipeline so it is fit per fold. Selection metric: average precision (PR-AUC), since defaults are ~10%. Decision threshold: maximise F1 on out-of-fold predictions of the chosen model.

In [ ]:
pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
cands={'baseline':Pipeline([('p',pre),('m',DummyClassifier(strategy='prior'))])}
for C_ in [0.01,0.1,1,10]:
    cands[f'logreg_C{C_}']=Pipeline([('p',pre),('m',LogisticRegression(C=C_,max_iter=1000))])
for d in [2,3]:
    for lr in [0.03,0.1]:
        cands[f'hgb_d{d}_lr{lr}']=Pipeline([('p',pre),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=lr,max_iter=200,early_stopping=False,random_state=SEED))])
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
oof={};rows=[]
for k,m in cands.items():
    p=cross_val_predict(m,X_tr,y_tr,cv=cv,method='predict_proba')[:,1]; oof[k]=p
    rows.append((k,average_precision_score(y_tr,p),roc_auc_score(y_tr,p),brier_score_loss(y_tr,p)))
cvres=pd.DataFrame(rows,columns=['model','cv_PR_AUC','cv_ROC_AUC','cv_brier']).sort_values('cv_PR_AUC',ascending=False)
cvres

## 5. Choice (CV only)
The model with the highest CV PR-AUC is chosen, and the threshold is the F1-maximising cut on its out-of-fold predictions. The test set has not been touched yet.

In [ ]:
best=cvres.iloc[0].model
p=oof[best]; ths=np.linspace(0.02,0.6,117)
f1s=[f1_score(y_tr,p>=t) for t in ths]; thr=float(ths[int(np.argmax(f1s))])
print('chosen:',best,'| threshold:',round(thr,3),'| CV F1:',round(max(f1s),3))
print('baseline CV PR-AUC (=prevalence):',cvres.set_index('model').loc['baseline','cv_PR_AUC'].round(3))

## 6. Refit on all training rows; score the test set once (headline)

In [ ]:
final=cands[best].fit(X_tr,y_tr)
pt=final.predict_proba(X_te)[:,1]; pred=pt>=thr
print(f'TEST ROC-AUC {roc_auc_score(y_te,pt):.3f} | PR-AUC {average_precision_score(y_te,pt):.3f} (prevalence {y_te.mean():.3f}) | brier {brier_score_loss(y_te,pt):.4f}')
print(f'at threshold {thr:.3f}: F1 {f1_score(y_te,pred):.3f}')
print(confusion_matrix(y_te,pred))